# U1 · Copy the Colab-era results from Google Drive to the Hugging Face results dataset
The first models (SciBERT/BERT NER, ViT, ResNet-50, caption SciBERT, fusion) were trained on Colab, so their
`metrics.json` / `history.csv` / `predictions.jsonl` live on Drive (`MyDrive/ResearchLens/results`). The Kaggle runs live in
the private HF dataset `researchlens-results`. This notebook **adds** the Drive results to that dataset (nothing is deleted),
so the R1 results notebook can show every model. CPU only, ~1 min. Needs the `HF_TOKEN` Colab secret (🔑).

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
SRC = "/content/drive/MyDrive/ResearchLens/results"
!ls {SRC}
!find {SRC} -name metrics.json | sort

In [ ]:
from huggingface_hub import HfApi
api = HfApi(); user = api.whoami()["name"]
repo = f"{user}/researchlens-results"
api.create_repo(repo, repo_type="dataset", private=True, exist_ok=True)
api.upload_folder(folder_path=SRC, path_in_repo="results", repo_id=repo, repo_type="dataset",
                  ignore_patterns=["*.npy", "*/report/*"], commit_message="Add Colab-era results from Drive")
print("done ->", f"https://huggingface.co/datasets/{repo}")
print(sorted({f.split('/')[1] + '/' + f.split('/')[2] for f in api.list_repo_files(repo, repo_type='dataset') if f.endswith('metrics.json')}))